# LeetCode 2148 — Count Elements With Strictly Smaller and Greater Elements bằng Python

Link đề: https://leetcode.com/problems/count-elements-with-strictly-smaller-and-greater-elements/ — Độ khó: **Easy**

Skeleton LeetCode:

```python
class Solution:
    def countElements(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu "**strictly** smaller/greater" (nhỏ hơn / lớn hơn *thực sự*, không có dấu bằng).
2. Biết dùng `min`, `max`, `count`, `Counter`, `sorted` và list comprehension / `sum` với điều kiện.
3. Viết 4 cách: brute force `O(n²)`, dùng `min/max` một lượt, đếm bằng `count`, và sort hai đầu.
4. Rút ra insight: phần tử thỏa mãn ⇔ **không phải min, cũng không phải max**.
5. Biết xử lý edge case: mảng chỉ có một giá trị khác nhau, hay chỉ có 1–2 phần tử.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Đếm số phần tử `x` trong `nums` sao cho **tồn tại** ít nhất một phần tử **nhỏ hơn hẳn** `x` và **tồn tại** ít nhất một phần tử **lớn hơn hẳn** `x`.

| nums | kết quả | giải thích |
|---|---|---|
| `[11,7,2,15]` | `2` | `7` có `2` nhỏ hơn và `11` lớn hơn; `11` có `7` nhỏ hơn và `15` lớn hơn |
| `[-3,3,3,90]` | `2` | hai số `3` đều có `-3` nhỏ hơn và `90` lớn hơn |
| `[5,5,5]` | `0` | không có phần tử nào nhỏ hơn hay lớn hơn hẳn |

Constraints: `1 <= nums.length <= 100`, `-10^5 <= nums[i] <= 10^5`.

Chú ý: mỗi **lần xuất hiện** đều được đếm, nên hai số `3` đóng góp `2`.

In [1]:
examples = [([11, 7, 2, 15], 2), ([-3, 3, 3, 90], 2), ([5, 5, 5], 0)]
for nums, expected in examples:
    print("nums =", nums, "| min =", min(nums), "| max =", max(nums), "| mong đợi =", expected)

nums = [11, 7, 2, 15] | min = 2 | max = 15 | mong đợi = 2
nums = [-3, 3, 3, 90] | min = -3 | max = 90 | mong đợi = 2
nums = [5, 5, 5] | min = 5 | max = 5 | mong đợi = 0


## 3. Kiến thức nền

### 3.1. `min(a)` và `max(a)`

`min(a)` / `max(a)` trả về phần tử nhỏ nhất / lớn nhất của list, chi phí `O(n)`. Với list rỗng sẽ báo `ValueError`, nhưng đề đảm bảo `n ≥ 1`.

In [2]:
a = [11, 7, 2, 15]
print(min(a), max(a))
print(min([5, 5, 5]), max([5, 5, 5]))

2 15
5 5


### 3.2. `any(...)` và `all(...)`

- `any(cond for x in a)`: `True` nếu **có ít nhất một** `x` thỏa `cond`.
- `all(cond for x in a)`: `True` nếu **mọi** `x` thỏa `cond`.

Câu "tồn tại phần tử nhỏ hơn" chính là `any(y < x for y in nums)`.

In [3]:
a = [11, 7, 2, 15]
x = 7
print(any(y < x for y in a), any(y > x for y in a))
x = 2
print(any(y < x for y in a), any(y > x for y in a))

True True
False True


### 3.3. `sum` với điều kiện

`sum(1 for x in a if điều_kiện)` đếm số phần tử thỏa điều kiện (vì mỗi phần tử thỏa góp `1`).

In [4]:
a = [11, 7, 2, 15]
print(sum(1 for x in a if x > 5))
print(sum(x > 5 for x in a))           # bool cộng như 0/1

3
3


### 3.4. `list.count` và `Counter`

`a.count(v)` đếm số lần `v` xuất hiện, `O(n)`. `Counter(a)` đếm tất cả giá trị trong **một** lượt, tra cứu `O(1)`.

In [5]:
from collections import Counter
a = [-3, 3, 3, 90]
print(a.count(3), a.count(100))
c = Counter(a)
print(c, c[3], c[100])

2 0
Counter({3: 2, -3: 1, 90: 1}) 2 0


### 3.5. `sorted` và so sánh hai đầu

Sau khi sort, phần tử nhỏ nhất ở `a[0]`, lớn nhất ở `a[-1]`. Các phần tử "ở giữa" (khác hai đầu) nằm gọn trong đoạn `a[i:j]` nếu ta biết vị trí kết thúc của nhóm min và bắt đầu của nhóm max.

In [6]:
a = sorted([-3, 3, 3, 90, -3])
print(a, a[0], a[-1])

[-3, -3, 3, 3, 90] -3 90


## 4. Ý tưởng nền tảng — tính bằng tay

`nums = [11, 7, 2, 15]`: nhỏ nhất `2`, lớn nhất `15`. Xét từng số:

- `11`: có `2` (hoặc `7`) nhỏ hơn, có `15` lớn hơn → đếm.
- `7`: có `2` nhỏ hơn, có `15` (hoặc `11`) lớn hơn → đếm.
- `2`: **không có** số nào nhỏ hơn → loại.
- `15`: **không có** số nào lớn hơn → loại.

Kết quả `2`. Nhận xét: chỉ cần nhìn xem số đó có phải **min** hoặc **max** hay không!

In [7]:
nums = [11, 7, 2, 15]
for x in nums:
    has_smaller = any(y < x for y in nums)
    has_greater = any(y > x for y in nums)
    print(f"x={x:3d}: có nhỏ hơn? {has_smaller!s:5s} | có lớn hơn? {has_greater!s:5s} | tính?", has_smaller and has_greater)

x= 11: có nhỏ hơn? True  | có lớn hơn? True  | tính? True
x=  7: có nhỏ hơn? True  | có lớn hơn? True  | tính? True
x=  2: có nhỏ hơn? False | có lớn hơn? True  | tính? False
x= 15: có nhỏ hơn? True  | có lớn hơn? False | tính? False


## 5. Cách 1 — Brute Force: với mỗi phần tử, quét tìm số nhỏ hơn và lớn hơn

Với mỗi `x`, duyệt toàn bộ mảng để xem có `y < x` và có `y > x` không.

In [8]:
def count_brute(nums):
    total = 0
    for x in nums:
        has_smaller = False
        has_greater = False
        for y in nums:
            if y < x:
                has_smaller = True
            if y > x:
                has_greater = True
        if has_smaller and has_greater:
            total += 1
    return total

for nums, expected in examples:
    print(count_brute(nums), expected)

2 2
2 2
0 0


Độ phức tạp: hai vòng lặp lồng nhau → **Time `O(n²)`**, **Space `O(1)`**. Với `n ≤ 100` vẫn chạy nhanh, nhưng nút thắt rõ ràng: ta lặp lại việc tìm "có số nhỏ hơn hay không" cho từng phần tử, trong khi câu trả lời chỉ phụ thuộc vào giá trị **min** và **max** toàn mảng.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `mn = min(nums)` và `mx = max(nums)`.

- `x` có số nhỏ hơn hẳn ⇔ `x > mn` (vì `mn` là số nhỏ nhất; nếu `x > mn` thì `mn` chính là một số nhỏ hơn hẳn; nếu `x == mn` thì không số nào nhỏ hơn).
- `x` có số lớn hơn hẳn ⇔ `x < mx`.

Vậy điều kiện đầy đủ: **`mn < x < mx`**. Số cần đếm = số phần tử nằm *nghiêm ngặt* giữa min và max. Nếu `mn == mx` (mọi số bằng nhau) thì không có `x` nào thỏa → `0`.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt với `min` / `max`

Tính `mn`, `mx` rồi đếm các `x` thỏa `mn < x < mx`.

In [9]:
def count_minmax(nums):
    mn, mx = min(nums), max(nums)
    return sum(1 for x in nums if mn < x < mx)

for nums, expected in examples:
    print(count_minmax(nums), expected)

2 2
2 2
0 0


In [10]:
from typing import List

class SolutionMinMax:
    def countElements(self, nums: List[int]) -> int:
        mn, mx = min(nums), max(nums)
        return sum(1 for x in nums if mn < x < mx)

print(SolutionMinMax().countElements([11, 7, 2, 15]))

2


Độ phức tạp: `min` + `max` + đếm = 3 lượt → **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Trừ đi số lần xuất hiện của min và max

Tổng số phần tử `n`, trừ số lần `min` xuất hiện và số lần `max` xuất hiện. Nếu `min == max` thì chúng là **cùng một nhóm**, kết quả `0` (đừng trừ hai lần!).

In [11]:
def count_subtract(nums):
    mn, mx = min(nums), max(nums)
    if mn == mx:
        return 0
    return len(nums) - nums.count(mn) - nums.count(mx)

for nums, expected in examples:
    print(count_subtract(nums), expected)

2 2
2 2
0 0


In [12]:
from typing import List

class SolutionSubtract:
    def countElements(self, nums: List[int]) -> int:
        mn, mx = min(nums), max(nums)
        if mn == mx:
            return 0
        return len(nums) - nums.count(mn) - nums.count(mx)

print(SolutionSubtract().countElements([-3, 3, 3, 90]))

2


Độ phức tạp: **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Sort rồi bỏ hai nhóm ở hai đầu

Sort mảng. Dùng hai con trỏ: `lo` nhảy qua mọi phần tử bằng `a[0]`, `hi` lùi qua mọi phần tử bằng `a[-1]`. Số phần tử còn lại giữa `lo` và `hi` là đáp án (nếu `lo > hi` thì `0`). Cách này luyện tư duy "cắt hai đầu" thường dùng khi bài đòi hỏi thứ tự.

In [13]:
def count_sorted(nums):
    a = sorted(nums)
    lo, hi = 0, len(a) - 1
    while lo < len(a) and a[lo] == a[0]:
        lo += 1
    while hi >= 0 and a[hi] == a[-1]:
        hi -= 1
    return max(0, hi - lo + 1)

for nums, expected in examples:
    print(count_sorted(nums), expected)

2 2
2 2
0 0


In [14]:
from typing import List

class SolutionSorted:
    def countElements(self, nums: List[int]) -> int:
        a = sorted(nums)
        lo, hi = 0, len(a) - 1
        while lo < len(a) and a[lo] == a[0]:
            lo += 1
        while hi >= 0 and a[hi] == a[-1]:
            hi -= 1
        return max(0, hi - lo + 1)

print(SolutionSorted().countElements([11, 7, 2, 15]))

2


Độ phức tạp: **Time `O(n log n)`** do sort, **Space `O(n)`** cho bản sao.

## 8. TRACE từng bước

Trace Cách 2 với `nums = [-3, 3, 3, 90]`: `mn = -3`, `mx = 90`.

| x | `mn < x` | `x < mx` | đếm? | total |
|---|---|---|---|---|
| -3 | `-3 < -3` sai | đúng | không | 0 |
| 3 | đúng | đúng | có | 1 |
| 3 | đúng | đúng | có | 2 |
| 90 | đúng | `90 < 90` sai | không | 2 |

In [15]:
def trace_minmax(nums):
    mn, mx = min(nums), max(nums)
    print(f"mn={mn}, mx={mx}")
    total = 0
    for x in nums:
        a, b = mn < x, x < mx
        if a and b:
            total += 1
        print(f"x={x}: mn<x là {a}, x<mx là {b} -> total={total}")
    return total

trace_minmax([-3, 3, 3, 90])

mn=-3, mx=90
x=-3: mn<x là False, x<mx là True -> total=0
x=3: mn<x là True, x<mx là True -> total=1
x=3: mn<x là True, x<mx là True -> total=2
x=90: mn<x là True, x<mx là False -> total=2


2

In [16]:
def trace_sorted(nums):
    a = sorted(nums)
    print("sau sort:", a)
    lo, hi = 0, len(a) - 1
    while lo < len(a) and a[lo] == a[0]:
        lo += 1
        print("  lo ->", lo)
    while hi >= 0 and a[hi] == a[-1]:
        hi -= 1
        print("  hi ->", hi)
    ans = max(0, hi - lo + 1)
    print(f"đoạn giữa a[{lo}..{hi}] có {ans} phần tử")
    return ans

trace_sorted([11, 7, 2, 15, 2, 15])
trace_sorted([5, 5, 5])

sau sort: [2, 2, 7, 11, 15, 15]
  lo -> 1
  lo -> 2
  hi -> 4
  hi -> 3
đoạn giữa a[2..3] có 2 phần tử
sau sort: [5, 5, 5]
  lo -> 1
  lo -> 2
  lo -> 3
  hi -> 1
  hi -> 0
  hi -> -1
đoạn giữa a[3..-1] có 0 phần tử


0

## 9. Code nộp LeetCode

In [17]:
from typing import List

class Solution:
    def countElements(self, nums: List[int]) -> int:
        mn, mx = min(nums), max(nums)
        return sum(1 for x in nums if mn < x < mx)

print(Solution().countElements([11, 7, 2, 15]), Solution().countElements([-3, 3, 3, 90]), Solution().countElements([5, 5, 5]))

2 2 0


## 10. Vì sao thuật toán đúng?

**Khẳng định:** `x` có phần tử nhỏ hơn hẳn ⇔ `x > min(nums)`.

- (⇐) Nếu `x > mn` thì chính phần tử `mn` (có trong mảng) nhỏ hơn hẳn `x`.
- (⇒) Nếu có `y < x` trong mảng thì `mn ≤ y < x`, nên `x > mn`.

Hoàn toàn tương tự: `x` có phần tử lớn hơn hẳn ⇔ `x < max(nums)`. Kết hợp hai điều kiện: `x` được đếm ⇔ `mn < x < mx`. Vì mỗi phần tử (kể cả trùng giá trị) được xét độc lập nên đếm đúng theo "mỗi lần xuất hiện".

Với Cách 3: các phần tử không được đếm đúng là những phần tử bằng `mn` hoặc bằng `mx`. Hai tập này **rời nhau** khi `mn < mx`, nên số phần tử được đếm là `n - count(mn) - count(mx)`; khi `mn == mx` chúng trùng nhau và đáp án là `0`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. `min`/`max` + đếm | `O(n)` | `O(1)` |
| 3. Trừ số lần xuất hiện | `O(n)` | `O(1)` |
| 4. Sort + hai con trỏ | `O(n log n)` | `O(n)` |

Worst case và average case bằng nhau cho cả bốn cách (luôn quét hết). Cách 2 và 3 duyệt mảng ba lượt (`min`, `max`, đếm); hằng số 3 không làm đổi bậc `O(n)`. Có thể gộp thành một lượt bằng cách cập nhật `mn`, `mx` song song, nhưng không cần thiết ở `n ≤ 100`.

## 12. Bộ test

Ví dụ đề, edge case (một phần tử, hai phần tử, mọi số bằng nhau, hai giá trị phân biệt, số âm) và đối chiếu mọi cách bằng input ngẫu nhiên có seed.

In [18]:
cases = [
    ([11, 7, 2, 15], 2),
    ([-3, 3, 3, 90], 2),
    ([5, 5, 5], 0),
    ([1], 0),
    ([1, 2], 0),
    ([1, 2, 3], 1),
    ([3, 3, 1, 1], 0),
    ([-5, -1, -1, 0, 4], 3),
    ([100000, -100000, 0], 1),
    ([2, 2, 2, 1, 3, 3], 3),
]
impls = [count_brute, count_minmax, count_subtract, count_sorted,
         Solution().countElements, SolutionMinMax().countElements,
         SolutionSubtract().countElements, SolutionSorted().countElements]
for nums, expected in cases:
    for f in impls:
        assert f(list(nums)) == expected, (f.__name__, nums, f(list(nums)))
print("Tất cả", len(cases), "ca x", len(impls), "cách đều đúng")

Tất cả 10 ca x 8 cách đều đúng


In [19]:
import random
random.seed(2148)
for _ in range(4000):
    n = random.randint(1, 15)
    nums = [random.randint(-5, 5) for _ in range(n)]
    want = count_brute(nums)
    for f in impls[1:]:
        assert f(list(nums)) == want, (f.__name__, nums)
print("4000 input ngẫu nhiên: mọi cách khớp brute force")

4000 input ngẫu nhiên: mọi cách khớp brute force


In [20]:
# tính chất: đáp án luôn nằm trong [0, n-2] (hoặc 0 nếu n<=2); thêm một số vào giữa tăng đáp án đúng 1
random.seed(1)
for _ in range(500):
    nums = [random.randint(-5, 5) for _ in range(random.randint(2, 10))]
    base = Solution().countElements(nums)
    assert 0 <= base <= max(0, len(nums) - 2)
    mn, mx = min(nums), max(nums)
    if mx - mn >= 2:
        bigger = nums + [mn + 1]
        assert Solution().countElements(bigger) == base + 1
print("tính chất 'thêm số nằm giữa thì tăng 1' đúng")

tính chất 'thêm số nằm giữa thì tăng 1' đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay vì `<` (không "strictly")

`mn <= x <= mx` đếm cả chính các số min và max, ra kết quả bằng `n`.

In [21]:
def wrong1(nums):
    mn, mx = min(nums), max(nums)
    return sum(1 for x in nums if mn <= x <= mx)

print(wrong1([11, 7, 2, 15]), "(sai) | đúng:", count_minmax([11, 7, 2, 15]))

4 (sai) | đúng: 2


### Lỗi 2: Chỉ loại một lần min và một lần max (dùng `remove`)

Nếu mảng có nhiều số bằng min hoặc max thì chỉ bỏ được một bản; các bản còn lại vẫn bị đếm nhầm. Ví dụ `[3,3,1,1]` phải ra `0`.

In [22]:
def wrong2(nums):
    a = list(nums)
    a.remove(min(a))
    if a:
        a.remove(max(a))
    return len(a)

print(wrong2([3, 3, 1, 1]), "(sai) | đúng:", count_minmax([3, 3, 1, 1]))

2 (sai) | đúng: 0


### Lỗi 3: Trừ `count(mn) + count(mx)` khi `mn == mx`

Khi mọi số bằng nhau, `count(mn)` và `count(mx)` đều bằng `n`, trừ hai lần cho `n - 2n = -n` (âm!).

In [23]:
def wrong3(nums):
    mn, mx = min(nums), max(nums)
    return len(nums) - nums.count(mn) - nums.count(mx)

print(wrong3([5, 5, 5]), "(sai, âm) | đúng:", count_subtract([5, 5, 5]))

-3 (sai, âm) | đúng: 0


### Lỗi 4: Dùng `set` làm mất số lần xuất hiện

`len(set(nums)) - 2` đếm số giá trị phân biệt nằm giữa, không đếm các bản trùng. Ví dụ 2 cần `2` nhưng ra `1`.

In [24]:
def wrong4(nums):
    return max(0, len(set(nums)) - 2)

print(wrong4([-3, 3, 3, 90]), "(sai) | đúng:", count_minmax([-3, 3, 3, 90]))

1 (sai) | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tồn tại phần tử nhỏ hơn / lớn hơn" → thường quy về so sánh với **min / max toàn cục**.
- Hãy đổi câu hỏi *có hay không* thành *so với một giá trị đại diện* (`min`, `max`, prefix min/max) để tránh vòng lặp lồng.
- Chú ý từ "strictly": ranh giới **không** được tính.
- Khi đếm "mỗi lần xuất hiện", đừng dùng `set`.
- Edge case hay quên: mảng chỉ có một giá trị duy nhất (`min == max`).
- Cách kiểm tra nhanh trong đầu: đáp án không thể vượt `n - 2` khi `n ≥ 2`.

## 15. Biến thể của bài

1. Đếm phần tử có **ít nhất một** phần tử nhỏ hơn **hoặc** lớn hơn (dễ hơn: loại khi mọi số bằng nhau).
2. Đếm phần tử có ít nhất **k** phần tử nhỏ hơn và ít nhất **k** phần tử lớn hơn (sort rồi bỏ `k` đầu và `k` cuối).
3. Bài mở rộng: với mỗi phần tử, đếm *bao nhiêu* phần tử nhỏ hơn nó ("How Many Numbers Are Smaller Than the Current Number").
4. Điều kiện nhỏ hơn/lớn hơn xét **chỉ về một phía** trong mảng (chỉ bên trái hoặc chỉ bên phải của nó): khi đó dùng mảng prefix-min / suffix-max thay cho min/max toàn cục.
5. Trả về **danh sách** các phần tử thỏa thay vì số lượng.

In [25]:
def count_at_least_k(nums, k):
    a = sorted(nums)
    return sum(1 for i, x in enumerate(a) if sum(1 for y in a if y < x) >= k and sum(1 for y in a if y > x) >= k)

def elements_between(nums):
    mn, mx = min(nums), max(nums)
    return [x for x in nums if mn < x < mx]

def count_smaller_each(nums):
    return [sum(1 for y in nums if y < x) for x in nums]

print(count_at_least_k([1, 2, 3, 4, 5], 2))
print(elements_between([-3, 3, 3, 90]))
print(count_smaller_each([8, 1, 2, 2, 3]))

1
[3, 3]
[4, 0, 1, 1, 3]


### Mở rộng: khi "nhỏ hơn / lớn hơn" chỉ xét một phía

Nếu đề đổi thành "có phần tử **bên trái** nhỏ hơn và **bên phải** lớn hơn", `min`/`max` toàn cục không còn đúng. Ta cần `prefix_min[i]` = min của `nums[:i]` và `suffix_max[i]` = max của `nums[i+1:]`. Ví dụ dưới đây dựng hai mảng này trong `O(n)` và đếm bằng một lượt; đây là bước nhảy tự nhiên từ bài này sang họ bài "prefix/suffix".

In [26]:
def count_left_smaller_right_greater(nums):
    n = len(nums)
    prefix_min = [float("inf")] * n      # min của nums[:i]
    for i in range(1, n):
        prefix_min[i] = min(prefix_min[i - 1], nums[i - 1])
    suffix_max = [float("-inf")] * n     # max của nums[i+1:]
    for i in range(n - 2, -1, -1):
        suffix_max[i] = max(suffix_max[i + 1], nums[i + 1])
    return sum(1 for i in range(n) if prefix_min[i] < nums[i] < suffix_max[i])

print(count_left_smaller_right_greater([11, 7, 2, 15]))
print(count_left_smaller_right_greater([1, 2, 3, 4]))
print(count_left_smaller_right_greater([4, 3, 2, 1]))

0
2
0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | với mỗi `x` tìm nhỏ hơn / lớn hơn | `O(n²)` | `O(1)` | hiểu đề, kiểm chứng |
| 2. `min`/`max` + đếm | đếm `mn < x < mx` | `O(n)` | `O(1)` | nộp bài, tối ưu & gọn |
| 3. Trừ số lần min, max | `n - count(mn) - count(mx)` | `O(n)` | `O(1)` | một dòng, nhớ ca `mn == mx` |
| 4. Sort + hai con trỏ | cắt hai nhóm ở hai đầu | `O(n log n)` | `O(n)` | khi cần thứ tự / mở rộng `k` |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `count_minmax([4, 4, 9, 1, 6, 6])` là bao nhiêu?

**Bài 2 (tính tay):** `nums = [7, 7, 7, 8]`. Tính `mn`, `mx`, đáp án và giải thích.

**Bài 3 (điền TODO):** hoàn thành `exercise_between(nums)` trả về **danh sách** các phần tử nằm nghiêm ngặt giữa min và max.

**Bài 4 (tự viết lại):** viết `exercise_subtract(nums)` theo Cách 3, nhớ ca `mn == mx`.

**Bài 5 (mini challenge):** viết `exercise_k(nums, k)` đếm phần tử có ít nhất `k` phần tử nhỏ hơn hẳn và ít nhất `k` phần tử lớn hơn hẳn.

In [27]:
# Bài 1: chạy để so với dự đoán
print(count_minmax([4, 4, 9, 1, 6, 6]))

4


In [28]:
def exercise_between(nums):
    # TODO: mn, mx = min(nums), max(nums); trả về list các x thỏa mn < x < mx
    pass

def exercise_subtract(nums):
    # TODO: nếu min == max trả 0; ngược lại len - count(min) - count(max)
    pass

def exercise_k(nums, k):
    # TODO: với mỗi x đếm số y < x và số y > x, cần cả hai >= k
    pass

# (không gọi khi chưa làm xong)

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Tìm min và max trước | `mn = 1`, `mx = 9` | các số 4,4,6,6 nằm giữa → `4` |
| 2 | Chỉ có hai giá trị khác nhau | `mn = 7`, `mx = 8` | không có số nằm giữa → `0` |
| 3 | List comprehension với điều kiện kép | `mn < x < mx` | `[x for x in nums if mn < x < mx]` |
| 4 | Xử lý ca `mn == mx` trước | `nums.count(...)` | `len(nums) - count(mn) - count(mx)` |
| 5 | Vòng lặp lồng được phép (n nhỏ) | Dùng `sum(1 for y in nums if y < x)` | cả hai tổng `>= k` thì cộng 1 |

## 19. Đáp án tham khảo cho bài tập

Bài 1: `4` (các số `4,4,6,6`). Bài 2: `mn = 7`, `mx = 8`; không có số nằm giữa nên `0`.

In [29]:
def exercise_between(nums):
    mn, mx = min(nums), max(nums)
    return [x for x in nums if mn < x < mx]

def exercise_subtract(nums):
    mn, mx = min(nums), max(nums)
    if mn == mx:
        return 0
    return len(nums) - nums.count(mn) - nums.count(mx)

def exercise_k(nums, k):
    total = 0
    for x in nums:
        smaller = sum(1 for y in nums if y < x)
        greater = sum(1 for y in nums if y > x)
        if smaller >= k and greater >= k:
            total += 1
    return total

assert count_minmax([4, 4, 9, 1, 6, 6]) == 4
assert exercise_between([7, 7, 7, 8]) == []
assert exercise_between([-3, 3, 3, 90]) == [3, 3]
assert exercise_subtract([5, 5, 5]) == 0
assert exercise_subtract([11, 7, 2, 15]) == 2
assert exercise_k([1, 2, 3, 4, 5], 2) == 1
assert exercise_k([1, 2, 3, 4, 5], 1) == 3
assert exercise_k([1, 2], 1) == 0
print("Đáp án bài tập đều đúng")

Đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
mn, mx = min(nums), max(nums)
return sum(1 for x in nums if mn < x < mx)
```

**Checklist:**

- [ ] Dùng `<` chặt, không phải `<=`?
- [ ] Mỗi lần xuất hiện của số ở giữa đều được đếm (không dùng `set`)?
- [ ] Ca mọi số bằng nhau trả về `0`?
- [ ] Ca `n = 1`, `n = 2` trả về `0`?
- [ ] Đã test số âm và giá trị biên `±10^5`?